<a href="https://colab.research.google.com/github/Gianluca-dot/Project-Deep-Learning-e-Reti-Neurali/blob/main/DEEP_LEARNING_E_PYTORCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## librerie importate

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchsummary import summary

import matplotlib.pyplot as plt
import numpy as np
import os
import random

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import confusion_matrix
import seaborn as sns
import json

import zipfile
import shutil

## seed per la riproducibilità

In [2]:
seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [3]:
torch.cuda.is_available()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## connessione a google drive per caricare il dataset:

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Percorsi definitivi configurati per installare il dataset su
# Google Colab, per ridurre il tempo dell'addestramento

zip_path = '/content/drive/MyDrive/ProgettoCibo/dataset.zip'
local_zip_path = '/content/dataset.zip'
extract_path = '/content/dataset_food'

print("1/2. Copia del file dataset.zip da Google Drive a Colab...")
shutil.copy(zip_path, local_zip_path)
print("Copia completata!")

print("2/2. Estrazione delle 14.000 immagini in corso... Attendi un attimo.")
with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Estrazione completata con successo nella cartella locale!")

# Eliminiamo il file zip temporaneo per liberare spazio sul disco di Colab
os.remove(local_zip_path)

# Verifica delle cartelle estratte
print("\nCartelle trovate all'interno di 'dataset_food':")
print(os.listdir(extract_path))

Mounted at /content/drive
1/2. Copia del file dataset.zip da Google Drive a Colab...
Copia completata!
2/2. Estrazione delle 14.000 immagini in corso... Attendi un attimo.
Estrazione completata con successo nella cartella locale!

Cartelle trovate all'interno di 'dataset_food':
['__MACOSX', 'dataset']


## classe transforms

In [5]:
class Transforms:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, img, *args, **kwargs):
        return self.transforms(image=np.array(img))['image']



transform = A.Compose([
            A.Resize(256, 256),
            A.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ToTensorV2(),
        ])




## carichiamo i dataset:
## validation e test

In [6]:
valset = torchvision.datasets.ImageFolder(root=os.path.join(extract_path, 'dataset', 'val'), transform=Transforms(transform))

testset = torchvision.datasets.ImageFolder(root=os.path.join(extract_path, 'dataset', 'test'),transform=Transforms(transform))

## carichiamo la rete pretrinata mobilenet_v3_large

In [7]:
# Caricamento del modello MobileNetV3-Large con i pesi preaddestrati
# Torno ad utilizzare 'pretrained=True' poiché la versione di torchvision non supporta MobileNetV3_Large_Weights.DEFAULT
model = torchvision.models.mobilenet_v3_large(pretrained=True)

print("Modello MobileNetV3-Large caricato con i pesi preaddestrati!")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 133MB/s]


Modello MobileNetV3-Large caricato con i pesi preaddestrati!


In [8]:
model.to(device)
summary(model, (3, 256, 256), device=device.type)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 128, 128]             432
       BatchNorm2d-2         [-1, 16, 128, 128]              32
         Hardswish-3         [-1, 16, 128, 128]               0
            Conv2d-4         [-1, 16, 128, 128]             144
       BatchNorm2d-5         [-1, 16, 128, 128]              32
              ReLU-6         [-1, 16, 128, 128]               0
            Conv2d-7         [-1, 16, 128, 128]             256
       BatchNorm2d-8         [-1, 16, 128, 128]              32
  InvertedResidual-9         [-1, 16, 128, 128]               0
           Conv2d-10         [-1, 64, 128, 128]           1,024
      BatchNorm2d-11         [-1, 64, 128, 128]             128
             ReLU-12         [-1, 64, 128, 128]               0
           Conv2d-13           [-1, 64, 64, 64]             576
      BatchNorm2d-14           [-1, 64,

In [9]:
for param in model.parameters():
    param.requires_grad = False

In [10]:
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 14)
model.to(device)

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [11]:
summary(model, (3, 256, 256))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 128, 128]             432
       BatchNorm2d-2         [-1, 16, 128, 128]              32
         Hardswish-3         [-1, 16, 128, 128]               0
            Conv2d-4         [-1, 16, 128, 128]             144
       BatchNorm2d-5         [-1, 16, 128, 128]              32
              ReLU-6         [-1, 16, 128, 128]               0
            Conv2d-7         [-1, 16, 128, 128]             256
       BatchNorm2d-8         [-1, 16, 128, 128]              32
  InvertedResidual-9         [-1, 16, 128, 128]               0
           Conv2d-10         [-1, 64, 128, 128]           1,024
      BatchNorm2d-11         [-1, 64, 128, 128]             128
             ReLU-12         [-1, 64, 128, 128]               0
           Conv2d-13           [-1, 64, 64, 64]             576
      BatchNorm2d-14           [-1, 64,

## creazione di un set di Data Augmentation personalizzato

In [12]:
augment = A.Compose([

    A.Resize(256, 256),

    A.HorizontalFlip(p=0.5),

    A.Rotate(limit=15, p=0.5),

    A.RandomBrightnessContrast(p=0.3),

    A.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),

    ToTensorV2(),

])

In [13]:
trainset = torchvision.datasets.ImageFolder(root=os.path.join(extract_path, 'dataset', 'train'), transform=Transforms(augment))

## carichiamo il dataset di train

In [14]:
class EarlyStopping:
    def __init__(self, save_path, patience=5, min_delta=0):

        self.save_path = save_path
        self.patience = patience
        self.min_delta = min_delta
        self.min_val_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, validation_loss, model):

        if self.min_val_loss is None:     #Prima epoca
          self.min_val_loss = validation_loss
          self.save_checkpoint(model)

        elif (self.min_val_loss - validation_loss) > self.min_delta: #Epoca con miglioramento
          self.min_val_loss = validation_loss
          self.save_checkpoint(model)
          self.counter = 0


        else:     #Nessun miglioramento
          self.counter +=1
          if self.counter >= self.patience:
            self.early_stop = True

    def save_checkpoint(self, model):
      torch.save(model.state_dict(), self.save_path)

In [15]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    processed_data = 0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device) #[128, 256, 256], [128, 14]

        optimizer.zero_grad()

        outputs = model(inputs) #[128, 14]
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        processed_data += len(inputs)

    return running_loss / processed_data

In [16]:
def test_epoch(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for i, data in enumerate(test_loader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [17]:
def train(model, train_loader, test_loader, criterion, optimizer, device, epochs=100, early_stopping=None):
    train_losses = []
    test_losses = []
    test_accuracies = []

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_accuracy = test_epoch(model, test_loader, criterion, device)

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {test_loss:.4f}, Validation Accuracy: {test_accuracy:.4f}')

        if early_stopping is not None:
            early_stopping(test_loss, model)
            if early_stopping.early_stop:
                print("Early stopping")
                break

        # Salvataggio del log ad ogni epoca (nuova aggiunta)
        to_save_json = {"train_losses":train_losses,
                        "test_losses":test_losses,
                        "test_accuracies":test_accuracies}
        with open(effnet_save_log, 'w') as f:
            json.dump(to_save_json, f)

    return train_losses, test_losses, test_accuracies

In [18]:
def plot_confusion_matrix(model, test_loader, device):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for i, data in enumerate(test_loader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)

            outputs = model(inputs)

            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10,7))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')

In [19]:
effnet_save_folder = "models/efficientnet/"
os.makedirs(effnet_save_folder, exist_ok=True)
effnet_save_file = os.path.join(effnet_save_folder, "model.pt")
effnet_save_log = os.path.join(effnet_save_folder, "log.json")

In [20]:
criterion = nn.CrossEntropyLoss()
criterion.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
early_stopping = EarlyStopping(effnet_save_file, patience=5, min_delta=0)
epochs = 100

In [21]:
do_train = True

In [22]:
batch_size = 16
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=False)
val_loader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=False)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=False)

In [23]:
if do_train:
    train_losses, test_losses, test_accuracies = train(model, train_loader, val_loader, criterion, optimizer, device, epochs, early_stopping)
    to_save_json = {"train_losses":train_losses,
                "test_losses":test_losses,
                "test_accuracies":test_accuracies}
    with open(effnet_save_log, 'w') as f:
      json.dump(to_save_json, f)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/100, Train Loss: 0.0686, Validation Loss: 0.0417, Validation Accuracy: 0.7960


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load(effnet_save_file))
with open(effnet_save_log, 'r') as f:
    log = json.load(f)

In [ ]:
if do_train:
  val_loss, val_accuracy = test_epoch(model, val_loader, criterion, device)
else:
  val_loss = model_log['test_losses'][-1]
  val_accuracy = model_log['test_accuracies'][-1]
print("Accuracy on val set: ", val_accuracy)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(log['train_losses'], label='Train Loss')
plt.plot(log['test_losses'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

In [ ]:
classes = trainset.classes
classes

In [ ]:
# print confusion matrix
if do_train:
  plot_confusion_matrix(model, val_loader, device)

##testiamo la rete sul set